# GLASS Interactive SI Shift

Display interactive latest-observed-count vs future SI-shift plots, with GLASS pathogen-antibiotic pairs highlighted.

In [101]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [102]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.pipelines import (
    build_glass_match_table,
    build_novel_prediction_table,
    build_si_analysis_paths,
    load_or_build_interactive_rank,
    load_si_and_clean_glass_list,
    plot_future_si_vs_si_shift,
    plot_latest_si_vs_future_si,
    summarize_pathogen_coverage,
)
from amr.si import plot_interactive_latest_observed_count_vs_future_si_shift

## Parameters

In [103]:
OUTPUT_DIR = PROJECT_ROOT / "outputs"
PARQUET_OUTPUT_DIR = OUTPUT_DIR / "parquets"
FIGURE_OUTPUT_DIR = OUTPUT_DIR / "figures"
PARQUET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = PARQUET_OUTPUT_DIR
ASSETS_DIR = PROJECT_ROOT / "assets"

ANTIBIOTIC_ANALYSIS_LEVEL = "antibiotic"  # antibiotic, Class, Simplified categories, High level categories, Origin
SI_PATHS = build_si_analysis_paths(PROJECT_ROOT, ANTIBIOTIC_ANALYSIS_LEVEL)
YEARLY_SCREENED_PATH = PARQUET_OUTPUT_DIR / "yearly_amr_screened_pairs.parquet"
YEARLY_SI_PATH = SI_PATHS["yearly_si"]
RANK_OUTPUT_PATH = SI_PATHS["rank_output"]

GLASS_XLSX_PATH = ASSETS_DIR / "GLASS list.xlsx"
GLASS_CLEANED_PATH = ASSETS_DIR / "glass_amr_cleaned.csv"
BACTERIA_SYNONYM_PATH = ASSETS_DIR / "bacteria_synonym_lookup.csv"
ANTIBIOTIC_SYNONYM_PATH = ASSETS_DIR / "antibiotic_synonym_lookup.csv"
ANTIBIOTIC_COARSENING_PATH = ASSETS_DIR / "antibiotic_coarsening_lookup.csv"

FIT_START_YEAR = 2000
FIT_END_YEAR = 2023
CURVE_FIT_END_YEAR = 2025
NUM_FUTURE_YEARS = 2
LOESS_FRAC = 0.6
LOESS_DEGREE = 1
TAIL_DAMPING_YEARS = None

MIN_LAST_AVAILABLE_COUNT = 1
MIN_COUNT_ABS_PCT_ERROR_MEAN = 0
MAX_COUNT_ABS_PCT_ERROR_MEAN = 100
MIN_COUNTS_BACKGROUND = 2
MIN_ABS_DELTA_SI = 0.01
MIN_SI_DELTA = None

DISPLAY_NAME = True
MAX_TEXT_LABELS = 300
WIDTH = 1400
HEIGHT = 900

## Load SI And Clean GLASS List

In [104]:
si_df, glass_pairs_df = load_si_and_clean_glass_list(
    YEARLY_SI_PATH,
    GLASS_XLSX_PATH,
    bacteria_synonym_path=BACTERIA_SYNONYM_PATH,
    antibiotic_synonym_path=ANTIBIOTIC_SYNONYM_PATH,
    antibiotic_coarsening_path=ANTIBIOTIC_COARSENING_PATH,
    glass_cleaned_path=GLASS_CLEANED_PATH,
)

si_source_df = pd.read_parquet(YEARLY_SCREENED_PATH)
si_pathogen_summary_df = summarize_pathogen_coverage(
    source_df=si_source_df,
    modeled_df=si_df,
    source_label="screened AMR input from notebook 01",
    modeled_label="filtered SI dataset from notebook 02",
)
display(si_pathogen_summary_df.query("status != 'present_in_both'"))
display(glass_pairs_df.head(20))

Loaded SI rows: 6,209 from /media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/outputs/parquets/yearly_si.parquet
Pathogens: 19; antibiotics/levels: 60
Saved cleaned GLASS rows: 89 to /media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/assets/glass_amr_cleaned.csv
screened AMR input from notebook 01 pathogens: 48
filtered SI dataset from notebook 02 pathogens: 19
Pathogens present in screened AMR input from notebook 01 but absent from filtered SI dataset from notebook 02: 29
Aggregatibacter actinomycetemcomitans, Bacillus anthracis, Bordetella pertussis, Borrelia burgdorferi, Brucella melitensis, Burkholderia cepacia, Burkholderia pseudomallei, Chlamydia pneumoniae, Chlamydia trachomatis, Clostridium perfringens, Coxiella burnetii, Cutibacterium acnes, Francisella tularensis, Legionella pneumophila, Moraxella catarrhalis, Mycobacterium abscessus, Mycobacterium avium, Mycobacterium leprae, Mycoplas

,pathogen,source_rows,source_publications,source_pairs,modeled_rows,modeled_publications,modeled_pairs,in_source,in_modeled,status
0,Aggregatibacter actinomycetemcomitans,97,114,21,0.0,0.0,0.0,True,False,missing_from_modeled
1,Bacillus anthracis,78,62,47,0.0,0.0,0.0,True,False,missing_from_modeled
2,Bordetella pertussis,71,97,25,0.0,0.0,0.0,True,False,missing_from_modeled
3,Borrelia burgdorferi,41,43,23,0.0,0.0,0.0,True,False,missing_from_modeled
4,Brucella melitensis,124,170,31,0.0,0.0,0.0,True,False,missing_from_modeled
5,Burkholderia cepacia,262,403,48,0.0,0.0,0.0,True,False,missing_from_modeled
6,Burkholderia pseudomallei,163,227,47,0.0,0.0,0.0,True,False,missing_from_modeled
7,Chlamydia pneumoniae,45,47,21,0.0,0.0,0.0,True,False,missing_from_modeled
8,Chlamydia trachomatis,150,154,38,0.0,0.0,0.0,True,False,missing_from_modeled
9,Clostridium perfringens,265,412,67,0.0,0.0,0.0,True,False,missing_from_modeled


,raw_pathogen,raw_antibiotic,pathogen,antibiotic,pathogen_standardized,antibiotic_standardized,Class,Simplified categories,High level categories,Origin
0,Acinetobacter spp.,amikacin,Acinetobacter baumannii,Amikacin,True,True,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
1,Acinetobacter spp.,Colistin,Acinetobacter baumannii,Colistin,True,False,Polymyxins,Miscellaneous,Other,Naturally occurring/modified
2,Acinetobacter spp.,doripenem,Acinetobacter baumannii,Doripenem,True,True,Carbapenems,Penems/Carbapenems,Beta lactam and related,Naturally occurring/modified
3,Acinetobacter spp.,Gentamicin,Acinetobacter baumannii,Gentamicin,True,False,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
4,Acinetobacter spp.,Imipenem,Acinetobacter baumannii,Imipenem/cilastatin,True,True,Carbapenems,Penems/Carbapenems,Beta lactam and related,Naturally occurring/modified
5,Acinetobacter spp.,meropenem,Acinetobacter baumannii,Meropenem,True,True,Carbapenems,Penems/Carbapenems,Beta lactam and related,Naturally occurring/modified
6,Acinetobacter spp.,minocycline,Acinetobacter baumannii,Minocycline,True,True,Tetracyclines,Tetracyclines and related,Tetracyclines and related,Naturally occurring/modified
7,Acinetobacter spp.,Tigecycline,Acinetobacter baumannii,Tigecycline,True,False,Glycylcyclines,Tetracyclines and related,Tetracyclines and related,Naturally occurring/modified
8,E. coli,Cefepime,Escherichia coli,Cefepime,True,False,Fourth-generation-cephalosporins,Cephalosporins,Beta lactam and related,Naturally occurring/modified
9,E. coli,cefotaxime,Escherichia coli,Cefotaxime,True,True,Third-generation-cephalosporins,Cephalosporins,Beta lactam and related,Naturally occurring/modified


## Build Or Load Rank Data

In [105]:
rank_df = load_or_build_interactive_rank(
    si_df,
    RANK_OUTPUT_PATH,
    source_path=YEARLY_SI_PATH,
    fit_start_year=FIT_START_YEAR,
    fit_end_year=FIT_END_YEAR,
    curve_fit_end_year=CURVE_FIT_END_YEAR,
    num_future_years=NUM_FUTURE_YEARS,
    loess_frac=LOESS_FRAC,
    loess_degree=LOESS_DEGREE,
    tail_damping_years=TAIL_DAMPING_YEARS,
    verbose=True,
)
display(rank_df)

Loaded cached rank rows: 241 from /media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/outputs/parquets/yearly_si_ranked_future.parquet


,pathogen,antibiotic,reference_year,future_year,num_future_years,fit_start_year,fit_end_year,latest_available_year,n_pair_years,current_si_source,...,post_fit_si_abs_pct_error_mean,loess_frac,loess_degree,tail_damping_years,status,error,current_rank,future_rank,rank_change,abs_rank_change
0,Salmonella enterica,Colistin,2023,2028,5,2000,2023,2026,18,observed,...,112.158384,0.6,1,None,ok,None,180.0,100.0,80.0,80.0
1,Staphylococcus aureus,Cefotaxime,2023,2028,5,2000,2023,2026,26,observed,...,200.607913,0.6,1,None,ok,None,144.0,75.0,69.0,69.0
2,Escherichia coli,Oxacillin,2023,2028,5,2000,2023,2026,26,observed,...,237.902892,0.6,1,None,ok,None,191.0,122.0,69.0,69.0
3,Acinetobacter baumannii,Tigecycline,2023,2028,5,2000,2023,2026,20,observed,...,26.521523,0.6,1,None,ok,None,62.0,130.0,-68.0,68.0
4,Escherichia coli,Vancomycin,2023,2028,5,2000,2023,2026,26,observed,...,44.163218,0.6,1,None,ok,None,197.0,131.0,66.0,66.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
236,Staphylococcus aureus,Levofloxacin,2023,2028,5,2000,2023,2026,25,observed,...,96.658940,0.6,1,None,ok,None,121.0,121.0,0.0,0.0
237,Campylobacter jejuni,Gentamicin,2023,2028,5,2000,2023,2026,26,observed,...,401.832314,0.6,1,None,ok,None,164.0,164.0,0.0,0.0
238,Pseudomonas aeruginosa,Sulbactam,2023,2028,5,2000,2023,2026,27,observed,...,201.966019,0.6,1,None,ok,None,231.0,231.0,0.0,0.0
239,Streptococcus pneumoniae,Ciprofloxacin,2023,2028,5,2000,2023,2026,27,observed,...,29.323023,0.6,1,None,ok,None,240.0,240.0,0.0,0.0


## Interactive Latest Count Vs Future SI Shift

In [106]:
latest_count_shift_result = plot_interactive_latest_observed_count_vs_future_si_shift(
    rank_df,
    glass_pairs_df=glass_pairs_df,
    min_last_available_count=MIN_LAST_AVAILABLE_COUNT,
    min_count_abs_pct_error_mean=MIN_COUNT_ABS_PCT_ERROR_MEAN,
    max_count_abs_pct_error_mean=MAX_COUNT_ABS_PCT_ERROR_MEAN,
    min_counts_background=MIN_COUNTS_BACKGROUND,
    min_abs_delta_si=MIN_ABS_DELTA_SI,
    min_si_delta=MIN_SI_DELTA,
    display_name=DISPLAY_NAME,
    max_text_labels=MAX_TEXT_LABELS,
    width=WIDTH,
    height=HEIGHT,
    pdf_path=FIGURE_OUTPUT_DIR / "interactive_latest_observed_count_vs_future_si_shift.pdf",
    show=True,
)
latest_count_shift_df = latest_count_shift_result["plot_df"]
print(f"Displayed rows: {len(latest_count_shift_df):,}; GLASS matches: {int(latest_count_shift_df['is_glass_pair'].sum()):,}")
if latest_count_shift_result["pdf_path"]:
    print(f"Saved PDF: {latest_count_shift_result['pdf_path']}")
elif latest_count_shift_result.get("pdf_error"):
    print(f"PDF export failed: {latest_count_shift_result['pdf_error']}")

/tmp/ipykernel_3116004/4085169871.py:1: RuntimeWarning: Could not save Plotly figure as PDF. Interactive plot display will continue. Plotly PDF export requires Kaleido and a working Chrome/Chromium install; try `choreo_get_chrome` or `plotly_get_chrome` if Chrome is the issue. Original error: ('The browser seemed to close immediately after starting.', 'You can set the `logging.Logger` level lower to see more output.', 'You may try installing a known working copy of Chrome by running ', '`$ choreo_get_chrome`.It may be your browser auto-updated and will now work upon restart. The browser we tried to start is located at /snap/bin/chromium.')
  latest_count_shift_result = plot_interactive_latest_observed_count_vs_future_si_shift(


Displayed rows: 175; GLASS matches: 40
PDF export failed: ('The browser seemed to close immediately after starting.', 'You can set the `logging.Logger` level lower to see more output.', 'You may try installing a known working copy of Chrome by running ', '`$ choreo_get_chrome`.It may be your browser auto-updated and will now work upon restart. The browser we tried to start is located at /snap/bin/chromium.')


## Display GLASS Matches In Plot

In [107]:
glass_plot_matches_df = build_glass_match_table(latest_count_shift_df)
display(glass_plot_matches_df)

,pathogen,antibiotic,latest_available_year,future_year,latest_available_count,future_predicted_count,latest_smoothed_SI,future_predicted_SI,future_minus_latest_smoothed_SI,post_fit_count_abs_pct_error_mean,glass_group_column
0,Escherichia coli,Ciprofloxacin,2026,2028,79.0,355.072393,0.666601,0.718543,0.051942,74.044079,antibiotic
1,Escherichia coli,Sulfamethoxazole/trimethoprim,2026,2028,53.0,180.716371,0.586901,0.624455,0.037554,56.379162,antibiotic
2,Klebsiella pneumoniae,Ceftazidime,2026,2028,46.0,160.269349,0.214151,0.246404,0.032254,41.976133,antibiotic
3,Escherichia coli,Meropenem,2026,2028,46.0,306.238310,0.506959,0.620519,0.113560,96.047829,antibiotic
4,Escherichia coli,Ceftazidime,2026,2028,46.0,159.447470,0.278695,0.333536,0.054841,46.945257,antibiotic
5,Klebsiella pneumoniae,Meropenem,2026,2028,44.0,217.697363,0.344506,0.381472,0.036966,69.546885,antibiotic
6,Acinetobacter baumannii,Colistin,2026,2028,43.0,115.265329,0.288376,0.295658,0.007282,32.238618,antibiotic
7,Klebsiella pneumoniae,Ciprofloxacin,2026,2028,38.0,118.759452,0.142441,0.155965,0.013524,42.117709,antibiotic
8,Escherichia coli,Ceftriaxone,2026,2028,37.0,168.971770,0.520383,0.589739,0.069356,72.603000,antibiotic
9,Pseudomonas aeruginosa,Meropenem,2026,2028,37.0,159.408697,0.313796,0.335605,0.021810,59.966796,antibiotic


## Novel Non-GLASS Predictions

These are plotted pairs that do not appear in the cleaned GLASS list, sorted as candidate novel predictions of interest.

In [108]:
novel_prediction_df = build_novel_prediction_table(latest_count_shift_df)
print(f"Novel non-GLASS predictions displayed: {len(novel_prediction_df):,}")
display(novel_prediction_df)

Novel non-GLASS predictions displayed: 135


,pathogen,antibiotic,latest_available_year,future_year,latest_available_count,future_predicted_count,latest_smoothed_SI,future_predicted_SI,future_minus_latest_smoothed_SI,post_fit_count_abs_pct_error_mean
0,Klebsiella pneumoniae,Ceftazidime/avibactam,2026,2028,26.0,160.309880,0.432001,0.573226,0.141225,68.747822
1,Escherichia coli,Vancomycin,2026,2028,12.0,62.571817,0.005130,0.126231,0.121101,72.050736
2,Escherichia coli,Meticillin,2026,2028,29.0,91.443838,0.166514,0.265095,0.098582,46.264682
3,Klebsiella pneumoniae,Nitrofurantoin,2026,2028,13.0,83.484503,0.112486,0.209358,0.096872,94.564060
4,Pseudomonas aeruginosa,Meticillin,2026,2028,23.0,119.341187,0.282823,0.376965,0.094143,77.314165
...,...,...,...,...,...,...,...,...,...,...
130,Acinetobacter baumannii,Ceftolozane/tazobactam,2026,2028,4.0,17.110334,-0.858746,-0.926092,-0.067346,94.153342
131,Staphylococcus aureus,Tigecycline,2026,2028,11.0,10.950949,-0.424617,-0.493093,-0.068476,26.465893
132,Streptococcus pneumoniae,Azithromycin,2026,2028,4.0,5.601126,-0.272864,-0.342921,-0.070057,21.058711
133,Acinetobacter baumannii,Ceftazidime,2026,2028,13.0,21.600294,-0.366875,-0.446410,-0.079535,38.352084


## Future SI Vs Future SI Shift

This uses the same y-axis definition as the latest-count plot: predicted future SI minus smoothed SI at the latest available year.

In [109]:
future_si_shift_result = plot_future_si_vs_si_shift(
    latest_count_shift_df,
    min_count_abs_pct_error_mean=MIN_COUNT_ABS_PCT_ERROR_MEAN,
    max_count_abs_pct_error_mean=MAX_COUNT_ABS_PCT_ERROR_MEAN,
    display_name=DISPLAY_NAME,
    max_text_labels=MAX_TEXT_LABELS,
    width=WIDTH,
    height=HEIGHT,
    pdf_path=FIGURE_OUTPUT_DIR / "future_si_vs_si_shift.pdf",
    show=True,
)
future_si_shift_fig = future_si_shift_result["figure"]
if future_si_shift_result["pdf_path"]:
    print(f"Saved PDF: {future_si_shift_result['pdf_path']}")
elif future_si_shift_result.get("pdf_error"):
    print(f"PDF export failed: {future_si_shift_result['pdf_error']}")

/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/amr/pipelines/glass_si.py:376: RuntimeWarning: Could not save Plotly figure as PDF. Interactive plot display will continue. Plotly PDF export requires Kaleido and a working Chrome/Chromium install; try `choreo_get_chrome` or `plotly_get_chrome` if Chrome is the issue. Original error: ('The browser seemed to close immediately after starting.', 'You can set the `logging.Logger` level lower to see more output.', 'You may try installing a known working copy of Chrome by running ', '`$ choreo_get_chrome`.It may be your browser auto-updated and will now work upon restart. The browser we tried to start is located at /snap/bin/chromium.')
  saved_pdf_path, pdf_error = _write_plotly_outputs(fig, html_path=html_path, pdf_path=pdf_path, width=width, height=height)


PDF export failed: ('The browser seemed to close immediately after starting.', 'You can set the `logging.Logger` level lower to see more output.', 'You may try installing a known working copy of Chrome by running ', '`$ choreo_get_chrome`.It may be your browser auto-updated and will now work upon restart. The browser we tried to start is located at /snap/bin/chromium.')
